# ПР-6. Прототип ШІ: прогноз часу доїзду швидкої до перехрестя
**Махун Владислав, група 242** · Smart Traffic Management System

Продовження ПР-5: гібрид ML-регресії та правил DP-04 / DP-10. Дані синтетичні; система не керує реальними світлофорами.

## 1. Підготовка даних
1200 синтетичних записів, seed=42. Ознаки: відстань, швидкість швидкої, черга, щільність, швидкість потоку, погода (0 — ясно, 1 — дощ, 2 — сильний дощ), година, день тижня. Ціль — час прибуття в секундах. Генератор моделює затримки та шум; це не виміряні реальні поїздки.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
rng=np.random.default_rng(42)
n=1200
df=pd.DataFrame({"distance_m":rng.uniform(70,650,n),"ambulance_speed_kmh":rng.uniform(15,75,n),"queue_cars":rng.integers(0,18,n),"density_pct":rng.uniform(10,95,n),"flow_speed_kmh":rng.uniform(8,65,n),"weather_code":rng.choice([0,1,2],n,p=[.65,.27,.08]),"hour":rng.integers(0,24,n),"weekday":rng.integers(0,7,n)})
v=np.maximum(8,.67*df.ambulance_speed_kmh+.33*df.flow_speed_kmh)
df["actual_time_s"]=(df.distance_m/(v/3.6)+.55*df.queue_cars+.035*df.density_pct+3*df.weather_code+np.where(df.hour.between(7,9)|df.hour.between(16,19),3.5,0)+rng.normal(0,2,n)).clip(lower=2).round(2)
print(df.shape)
display(df.head())

(1200, 9)


,distance_m,ambulance_speed_kmh,queue_cars,density_pct,flow_speed_kmh,weather_code,hour,weekday,actual_time_s
0,518.894508,48.985239,11,80.483797,19.687884,1,13,0,61.74
1,324.549495,48.054526,10,60.498159,53.955195,0,0,2,29.93
2,567.986794,64.712815,3,30.812654,14.031428,0,20,2,42.25
3,474.473457,57.631966,14,73.382375,43.022367,0,2,2,41.25
4,124.622862,16.594666,12,17.180876,50.532334,0,23,6,26.00


## 2. Навчання регресії та квантильних моделей
Випадковий поділ 80/20 використано лише для навчального синтетичного прототипу. У реальній системі потрібен часовий поділ історичних поїздок. Інтервал 10–90% не є гарантією 80% покриття; його треба перевіряти.

In [2]:
features=[c for c in df if c!="actual_time_s"]
X_train,X_test,y_train,y_test=train_test_split(df[features],df.actual_time_s,test_size=.2,random_state=42)
models={}
for name,loss,alpha in [("mean","squared_error",.9),("low","quantile",.1),("high","quantile",.9)]:
    model=GradientBoostingRegressor(loss=loss,alpha=alpha,n_estimators=170,max_depth=3,learning_rate=.06,random_state=42)
    model.fit(X_train,y_train)
    models[name]=model
pred=models["mean"].predict(X_test)
low=models["low"].predict(X_test)
high=models["high"].predict(X_test)
lo=np.minimum(low,high); hi=np.maximum(low,high)
mae=mean_absolute_error(y_test,pred)
baseline=mean_absolute_error(y_test,X_test.distance_m/(np.maximum(X_test.ambulance_speed_kmh,1)/3.6))
coverage=np.mean((y_test>=lo)&(y_test<=hi))
print(f"MAE ML: {mae:.2f} s; baseline d/v: {baseline:.2f} s; interval coverage: {coverage:.1%}")

MAE ML: 3.08 s; baseline d/v: 12.17 s; interval coverage: 65.8%


## 3. Логіка рішення DP-04 та захист DP-10
`confidence_score = clip(1 − ширина_інтервалу/100, 0, 1)` — **евристичний бал**, а не калібрована ймовірність. Автоматичне керування світлофорами за цим балом заборонено без реальної калібровки, валідації та перевірок безпеки. GPS старше 5 с → ескалація. Поріг T < 15 с та score > 0.90 повторює логіку ПР-5 лише в симуляції.

In [3]:
def predict_eta(payload):
    gps_age=payload.get("gps_age_s")
    if gps_age is None or gps_age<0 or gps_age>5:
        return {"decision":"ESCALATE_STALE_GPS", "predicted_s":None, "confidence_score":None}
    if any(k not in payload or pd.isna(payload[k]) for k in features):
        return {"decision":"ESCALATE_MISSING_DATA", "predicted_s":None, "confidence_score":None}
    row=pd.DataFrame([{k:payload[k] for k in features}])
    t=float(models["mean"].predict(row)[0]); l=float(models["low"].predict(row)[0]); h=float(models["high"].predict(row)[0])
    l,h=min(l,h),max(l,h)
    score=float(np.clip(1-(h-l)/100,0,1))
    decision="RECOMMEND_GREEN" if t<15 and score>.90 else "NO_AUTO_GREEN"
    return {"predicted_s":round(t,2),"interval_s":[round(l,2),round(h,2)],"confidence_score":round(score,3),"decision":decision,"command_sent":False}
print("Prototype loaded; no traffic-light commands are sent.")

Prototype loaded; no traffic-light commands are sent.


## 4. Шість різних тестових випадків
Критерій коректності числового прогнозу: абсолютна різниця з незалежним синтетичним еталоном ≤ 5 с. Для простроченого GPS перевіряємо також обов’язкову ескалацію.

In [4]:
cases=pd.DataFrame({"distance_m":[110,250,480,170,320,90],"ambulance_speed_kmh":[65,45,28,60,35,70],"queue_cars":[0,5,15,2,9,0],"density_pct":[15,55,90,25,80,12],"flow_speed_kmh":[55,35,12,50,17,62],"weather_code":[0,1,2,0,1,0],"hour":[11,8,18,14,17,12],"weekday":[1,2,4,3,0,5],"gps_age_s":[1,2,3,8,2,1]})
v=np.maximum(8,.67*cases.ambulance_speed_kmh+.33*cases.flow_speed_kmh)
cases["expected_s"]=(cases.distance_m/(v/3.6)+.55*cases.queue_cars+.035*cases.density_pct+3*cases.weather_code+np.where(cases.hour.between(7,9)|cases.hour.between(16,19),3.5,0)).round(1)
results=[]
for _,r in cases.iterrows():
    output=predict_eta(r.to_dict())
    numerical_ok=output["predicted_s"] is not None and abs(output["predicted_s"]-r.expected_s)<=5
    safety_ok=(r.gps_age_s<=5 or output["decision"]=="ESCALATE_STALE_GPS")
    results.append({"expected_s":r.expected_s,"predicted_s":output["predicted_s"],"decision":output["decision"],"correct":bool(safety_ok and (numerical_ok or r.gps_age_s>5))})
display(pd.DataFrame(results))

,expected_s,predicted_s,decision,correct
0,6.9,6.44,NO_AUTO_GREEN,True
1,32.8,30.41,NO_AUTO_GREEN,True
2,97.0,96.60,NO_AUTO_GREEN,True
3,12.8,NaN,ESCALATE_STALE_GPS,True
4,53.9,47.97,NO_AUTO_GREEN,False
5,5.2,5.80,NO_AUTO_GREEN,True


## 5. Інтеграція
GPS / камери / IoT / метео → **Модуль збору та підготовки №2** → **ШІ-ядро №3 (цей прототип)** → `predicted_s`, `interval_s`, `confidence_score`, `decision` → **Модуль рішень №4 (DP-04/DP-10)** → **Контролер №5 / Інтерфейс диспетчера №1 / Журнал аудиту №7**.

У цій демонстрації вихід — Python-словник, який у реальній системі може бути серіалізований як JSON. Фізичних команд світлофору немає.

**Обмеження:** синтетичні дані, некалібрований score, відсутні інтеграція з реальними сенсорами, тестування на ДТП та сертифікація безпеки.